# Appendix Notebook A6: Case B for Macro-Financial Regime Learning

This appendix notebook is the standardised review-paper version of the original local working file for Module 4.

This notebook accompanies Module 4 Case B and studies a macro-financial learning task with both classical and quantum model families.

## Reading Guide

The workflow includes data retrieval, macro-asset alignment, train-test splitting, benchmark fitting, quantum model evaluation, and final comparison plots.

All file names in `review_paper/notebooks/` follow a common naming convention so the paper appendix can reference them consistently.

# Case B: Notebook Setup

This notebook defines the environment and core configuration for the macro-financial learning case.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score

import pennylane as qml
from tqdm import tqdm


def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'review_paper' / 'assets').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def load_local_market_frame(csv_path: Path) -> pd.DataFrame:
    raw = pd.read_csv(csv_path, header=[0, 1])
    dates = pd.to_datetime(raw.iloc[1:, 0])
    close = pd.to_numeric(raw.iloc[1:, 1], errors='coerce')
    return pd.DataFrame({'Close': close.to_numpy()}, index=dates).dropna().sort_index()


def load_local_fred_series(csv_path: Path, series_name: str) -> pd.Series:
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    return pd.Series(df['value'].to_numpy(), index=df['date'], name=series_name).sort_index()

ROOT = locate_repo_root(Path.cwd())
RESULT_DIR = ROOT / 'review_paper' / 'assets' / 'module_04_qml' / 'data'
RESULT_DIR.mkdir(parents=True, exist_ok=True)


## Asset-Side Inputs

The primary financial series are pulled and prepared before macro factors are introduced.

In [ ]:
# ==============================
# User-configurable settings
# ==============================
TICKER = 'SPY'
START_DATE = '2024-01-01'
END_DATE   = '2025-12-31'
SPY_CSV = RESULT_DIR / f'{TICKER}_{START_DATE}_{END_DATE}_daily.csv'
FRED_FILES = {
    'FEDFUNDS': RESULT_DIR / f'FEDFUNDS_{START_DATE}_{END_DATE}.csv',
    'CPI': RESULT_DIR / f'CPIAUCSL_{START_DATE}_{END_DATE}.csv',
    'INDPRO': RESULT_DIR / f'INDPRO_{START_DATE}_{END_DATE}.csv',
}

SEED = 42
np.random.seed(SEED)

# ------------------------------
# Label design: volatility regime
# ------------------------------
VOL_WINDOW = 20          # realised vol window (20 trading days ~ 1 month)
VOL_ANNUALIZE = True
VOL_THRESHOLD_Q = 0.70   # :(70%),

# ------------------------------
# Train/test split (time-based)
# ------------------------------
TRAIN_RATIO = 0.7        # 70%,30%(shuffle)

# ------------------------------
# Quantum model sizes (NISQ-ish)
# ------------------------------
N_QUBITS = 5
N_LAYERS_QNN = 3

# QSVC kernel compute budget ()
MAX_QSVC_TRAIN = 250
MAX_QSVC_TEST  = 250

# QNN training budget
MAX_QNN_TRAIN = 400
MAX_QNN_TEST  = 400
EPOCHS = 15
LR_QNN = 0.15

# Metrics threshold
ALPHA_PROBA_THRESHOLD = 0.5

print('Local market CSV:', SPY_CSV)
print('Local FRED files:', FRED_FILES)


## Shared Evaluation Logic

Reusable performance metrics are defined so every model is judged consistently.

In [ ]:
def eval_binary(y_true, proba, thr=0.5):
    y_pred = (proba >= thr).astype(int)
    return {
        "AUC": float(roc_auc_score(y_true, proba)) if len(np.unique(y_true)) > 1 else np.nan,
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "BalancedAcc": float(balanced_accuracy_score(y_true, y_pred)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0))
    }

def extract_close(df):
    """Ensure the local SPY CSV is parsed into a clean single Close series."""
    if isinstance(df.columns, pd.MultiIndex):
        close = df["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        return close
    return df["Close"]

## Market Data Download

The asset-level data window is established and checked for missing observations.

In [ ]:
if not SPY_CSV.exists():
    raise FileNotFoundError(f'Missing expected local market CSV: {SPY_CSV}')

spy = load_local_market_frame(SPY_CSV)

# log returns
spy['r'] = np.log(spy['Close'] / spy['Close'].shift(1))

# realised vol (rolling std)
spy['vol'] = spy['r'].rolling(VOL_WINDOW).std()
if VOL_ANNUALIZE:
    spy['vol'] = spy['vol'] * np.sqrt(252)

# drawdown
spy['cum'] = (1 + spy['r'].fillna(0)).cumprod()
spy['peak'] = spy['cum'].cummax()
spy['drawdown'] = (spy['cum'] / spy['peak']) - 1.0

spy = spy.dropna().copy()
print('SPY rows:', len(spy))
spy.head()


## Macro Data Retrieval

Macro-financial covariates are collected and aligned with the market series.

In [ ]:
missing = [name for name, path in FRED_FILES.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing expected local FRED CSVs: ' + ', '.join(missing))

macro = pd.DataFrame({
    'FEDFUNDS': load_local_fred_series(FRED_FILES['FEDFUNDS'], 'FEDFUNDS'),
    'CPI': load_local_fred_series(FRED_FILES['CPI'], 'CPI'),
    'INDPRO': load_local_fred_series(FRED_FILES['INDPRO'], 'INDPRO'),
})

macro.index = pd.to_datetime(macro.index)
macro = macro.sort_index()

# resample to daily, forward fill
macro = macro.resample('D').ffill()

# macro transforms (log-diff)
macro['dCPI'] = np.log(macro['CPI']).diff()
macro['dIP']  = np.log(macro['INDPRO']).diff()

macro = macro.dropna().copy()
print('Macro rows:', len(macro))
macro.tail()


## Dataset Alignment and Cleaning

All features are merged into a single modelling table with compatible timestamps and naming.

In [ ]:
# Ensure both are single-level columns
spy2 = spy.copy()
spy2.columns = [str(c) for c in spy2.columns]

macro2 = macro.copy()
macro2.columns = [str(c) for c in macro2.columns]

df = spy2.join(macro2[["FEDFUNDS", "dCPI", "dIP"]], how="inner").copy()

# Keep only modelling columns first, then dropna once
FEATURES = ["vol", "drawdown", "FEDFUNDS", "dCPI", "dIP"]

# Label: volatility regime
# y_t = 1{ vol_t >= q(train_vol) }  (threshold learned from training window only)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["vol"]).copy()

print("Joined rows:", len(df))
df.head()

## Train-Test Construction

The sample is split in time order to preserve the forecasting structure of the exercise.

In [ ]:
# time-based split
n = len(df)
split = int(n * TRAIN_RATIO)

df_train = df.iloc[:split].copy()
df_test  = df.iloc[split:].copy()

# threshold learned from training data only
thr_vol = df_train["vol"].quantile(VOL_THRESHOLD_Q)

df_train["y"] = (df_train["vol"] >= thr_vol).astype(int)
df_test["y"]  = (df_test["vol"]  >= thr_vol).astype(int)

print("Vol threshold (train quantile):", float(thr_vol))
print("Train pos-rate:", df_train["y"].mean(), "Test pos-rate:", df_test["y"].mean())

# Build X/y
X_train_raw = df_train[FEATURES].values
y_train = df_train["y"].values.astype(int)

X_test_raw  = df_test[FEATURES].values
y_test  = df_test["y"].values.astype(int)

# Standardize (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print("Shapes:", X_train.shape, X_test.shape)

## Classical Benchmark Models

Classical reference models are estimated first to create a grounded comparison point.

In [ ]:
results = []

# Logistic Regression
lr = LogisticRegression(max_iter=2000, random_state=SEED)
lr.fit(X_train, y_train)
proba_lr = lr.predict_proba(X_test)[:, 1]
results.append({"model": "LR", **eval_binary(y_test, proba_lr, ALPHA_PROBA_THRESHOLD)})

# SVC (RBF)
svc = SVC(kernel="rbf", probability=True, random_state=SEED)
svc.fit(X_train, y_train)
proba_svc = svc.predict_proba(X_test)[:, 1]
results.append({"model": "SVC(RBF)", **eval_binary(y_test, proba_svc, ALPHA_PROBA_THRESHOLD)})

# MLP
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=600, random_state=SEED)
mlp.fit(X_train, y_train)
proba_mlp = mlp.predict_proba(X_test)[:, 1]
results.append({"model": "NN(MLP)", **eval_binary(y_test, proba_mlp, ALPHA_PROBA_THRESHOLD)})

pd.DataFrame(results)

## Variational Quantum Model

The first quantum branch maps scaled features into a variational circuit.

In [ ]:
# We'll map features -> angles with tanh squashing for stability.
# X is already standardized; we further squash into [-pi, pi].
def squash_angles(x):
    return np.pi * np.tanh(x)

# Choose first N_QUBITS features if FEATURES > N_QUBITS
D = X_train.shape[1]
if D < N_QUBITS:
    raise ValueError(f"Need at least {N_QUBITS} features, but got {D}.")

def feature_map(x):
    # x: length N_QUBITS vector
    ang = squash_angles(x)
    for i in range(N_QUBITS):
        qml.RY(ang[i], wires=i)
        qml.RZ(0.5 * ang[i], wires=i)
    # simple entanglement ring
    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i+1])
    qml.CNOT(wires=[N_QUBITS-1, 0])

dev_feat = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_feat)
def q_features(x):
    feature_map(x)
    # expectation features: Z on each qubit
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

def build_qfeat_matrix(X_in, n_max=None):
    Xs = X_in[:, :N_QUBITS]
    if n_max is not None:
        Xs = Xs[:n_max]
    feats = []
    for x in tqdm(Xs, desc="Q-features", leave=False):
        feats.append(q_features(x))
    return np.array(feats, dtype=float)

# Build quantum features (train/test)
QTRAIN_MAX = min(1200, len(X_train))
QTEST_MAX  = min(1200, len(X_test))

Xqf_train = build_qfeat_matrix(X_train, n_max=QTRAIN_MAX)
yqf_train = y_train[:QTRAIN_MAX]

Xqf_test  = build_qfeat_matrix(X_test, n_max=QTEST_MAX)
yqf_test  = y_test[:QTEST_MAX]

# Logistic head on quantum-derived features
qlr = LogisticRegression(max_iter=2000, random_state=SEED)
qlr.fit(Xqf_train, yqf_train)
proba_qlr = qlr.predict_proba(Xqf_test)[:, 1]
results.append({"model": "QLR(quantum-features)", **eval_binary(yqf_test, proba_qlr, ALPHA_PROBA_THRESHOLD)})

pd.DataFrame(results)

## Quantum Kernel Model

A kernel-based quantum model is evaluated on the same macro-financial task.

In [ ]:
# Quantum kernel: K(x_i, x_j) = |<phi(x_i)|phi(x_j)>|^2
dev_k = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_k)
def state_phi(x):
    feature_map(x)
    return qml.state()

def qkernel(x1, x2):
    s1 = state_phi(x1)
    s2 = state_phi(x2)
    return np.abs(np.vdot(s1, s2))**2

# Subsample for tractability
ntr = min(len(X_train), MAX_QSVC_TRAIN)
nte = min(len(X_test),  MAX_QSVC_TEST)

Xq_tr_sub = X_train[:ntr, :N_QUBITS]
y_tr_sub  = y_train[:ntr]
Xq_te_sub = X_test[:nte, :N_QUBITS]
y_te_sub  = y_test[:nte]

print("QSVC subset sizes:", ntr, nte)

# Build Gram matrices with ONE progress bar
K_train = np.zeros((ntr, ntr))
for i in tqdm(range(ntr), desc="QSVC Gram (train)", leave=True):
    for j in range(i, ntr):
        val = qkernel(Xq_tr_sub[i], Xq_tr_sub[j])
        K_train[i, j] = val
        K_train[j, i] = val

K_test = np.zeros((nte, ntr))
for i in tqdm(range(nte), desc="QSVC Gram (test)", leave=True):
    for j in range(ntr):
        K_test[i, j] = qkernel(Xq_te_sub[i], Xq_tr_sub[j])

qsvc = SVC(kernel="precomputed", probability=True, random_state=SEED)
qsvc.fit(K_train, y_tr_sub)

proba_qsvc = qsvc.predict_proba(K_test)[:, 1]
results.append({"model": "QSVC(quantum-kernel)", **eval_binary(y_te_sub, proba_qsvc, ALPHA_PROBA_THRESHOLD)})

pd.DataFrame(results)

## Quantum Neural Network Model

A variational classifier is trained as the second expressive quantum benchmark.

In [ ]:
dev_qnn = qml.device("default.qubit", wires=N_QUBITS)

def variational_block(theta):
    # theta shape: (N_LAYERS_QNN, N_QUBITS, 3)
    for l in range(N_LAYERS_QNN):
        for i in range(N_QUBITS):
            qml.Rot(theta[l, i, 0], theta[l, i, 1], theta[l, i, 2], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[N_QUBITS-1, 0])

@qml.qnode(dev_qnn)
def qnn_forward(x, theta):
    feature_map(x)
    variational_block(theta)
    return qml.expval(qml.PauliZ(0))

def sigmoid_pl(z):
    # IMPORTANT: use qml.numpy for autograd compatibility
    return 1.0 / (1.0 + qml.numpy.exp(-z))

def bce_loss_pl(y_true, y_prob, eps=1e-9):
    y_prob = qml.numpy.clip(y_prob, eps, 1-eps)
    return -(y_true*qml.numpy.log(y_prob) + (1-y_true)*qml.numpy.log(1-y_prob))

# Subsample (speed)
ntr = min(MAX_QNN_TRAIN, len(X_train))
nte = min(MAX_QNN_TEST,  len(X_test))

Xq_tr = X_train[:ntr, :N_QUBITS]
y_tr  = y_train[:ntr]
Xq_te = X_test[:nte, :N_QUBITS]
y_te  = y_test[:nte]

# init
rng = np.random.RandomState(SEED)
theta0 = 0.05 * rng.normal(size=(N_LAYERS_QNN, N_QUBITS, 3))
theta = qml.numpy.array(theta0, requires_grad=True)

def objective(theta_var):
    # compute logits for all training points
    logits = qml.numpy.stack([qnn_forward(x, theta_var) for x in Xq_tr])
    probs  = sigmoid_pl(2.0 * logits)  # scaling
    return qml.numpy.mean(bce_loss_pl(y_tr, probs))

opt = qml.GradientDescentOptimizer(stepsize=LR_QNN)

for ep in tqdm(range(1, EPOCHS+1), desc="QNN epochs", leave=True):
    theta, loss_val = opt.step_and_cost(objective, theta)
    if ep == 1 or ep % 5 == 0:
        print(f"Epoch {ep:02d} | loss = {float(loss_val):.5f}")

# Evaluate (no progress bar)
logits_te = np.array([qnn_forward(x, theta) for x in Xq_te], dtype=float)
proba_qnn = 1.0 / (1.0 + np.exp(-2.0 * logits_te))
results.append({"model": "QNN(variational)", **eval_binary(y_te, proba_qnn, ALPHA_PROBA_THRESHOLD)})

pd.DataFrame(results)

## Result Consolidation

The notebook aggregates metrics and intermediate outputs into a final comparison table.

In [ ]:
res = pd.DataFrame(results)

order = [
    'LR',
    'QLR(quantum-features)',
    'SVC(RBF)',
    'QSVC(quantum-kernel)',
    'NN(MLP)',
    'QNN(variational)',
]
res['model'] = pd.Categorical(res['model'], categories=order, ordered=True)
res = res.sort_values('model').reset_index(drop=True)
summary_path = RESULT_DIR / 'module_04_case_b_results.csv'
res.to_csv(summary_path, index=False, float_format='%.6f')
print(f'Results saved to: {summary_path}')
res


## Interpretation Plotting

The last section prepares final visual summaries for appendix-style discussion.

In [ ]:
tmp = df.copy()
tmp["y"] = (tmp["vol"] >= thr_vol).astype(int)

plt.figure(figsize=(11,4))
plt.plot(tmp.index, tmp["vol"], label="Realised vol")
plt.axhline(thr_vol, linestyle="--", label=f"Threshold q={VOL_THRESHOLD_Q:.2f}")
plt.title("SPY realised volatility regime label (1=high vol)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print("Label balance (full):", tmp["y"].mean())

## Interpretation Note

Case B remains reproducible in notebook form, while the main article concentrates on comparative insight and deployment relevance.